In [1]:
!pip install --no-cache-dir "gymnasium[atari,accept-rom-license]"
!pip install torch torchvision
!pip install opencv-python matplotlib

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 434.7/434.7 kB 11.9 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
INFO: pip is looking at multiple versions of shimmy[atari] to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 952.1/952.1 kB 75.8 MB/s eta 0:00:00
  Attempting uninstall: gymnasium
    Found existing installation: gymnasium 0.29.0
    Uninstalling gymnasium-0.29.0:
      Successfully uninstalled gymnasium-0.29.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
stable-baselines3 2.1.0 requires gymnasium<0.30,>=0.28.1, but you have gymnasium 1.2.3 which is incompatible.
kaggle-environments 1.18.0 requires gymnasium==0.29.0, but you have gymnasium 1.2.3 which is incompat

In [2]:
# =============================================================================
# COMPLETE CODE - LOAD AND RUN BASELINE AGENT
# =============================================================================

import gymnasium as gym
import ale_py
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from collections import deque
from dataclasses import dataclass
import cv2

gym.register_envs(ale_py)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Config
@dataclass
class DQNConfig:
    env_name: str = "ALE/Seaquest-v5"
    total_episodes: int = 5000
    learning_rate: float = 0.00025
    gamma: float = 0.99
    epsilon_start: float = 1.0
    epsilon_end: float = 0.01
    epsilon_decay: float = 0.00001
    buffer_size: int = 100000
    batch_size: int = 32
    min_buffer_size: int = 10000
    target_update_freq: int = 1000
    frame_height: int = 84
    frame_width: int = 84
    frame_stack: int = 4

# Preprocessing
class FramePreprocessor:
    def __init__(self, h=84, w=84):
        self.h, self.w = h, w
    def preprocess(self, frame):
        gray = cv2.cvtColor(frame, cv2.COLOR_RGB2GRAY)
        return cv2.resize(gray, (self.w, self.h)).astype(np.float32) / 255.0

class FrameStacker:
    def __init__(self, size=4):
        self.frames = deque(maxlen=size)
    def reset(self, frame):
        for _ in range(4): self.frames.append(frame)
        return np.stack(self.frames, axis=0)
    def add_frame(self, frame):
        self.frames.append(frame)
        return np.stack(self.frames, axis=0)

# Network
class DQN(nn.Module):
    def __init__(self, input_shape, n_actions):
        super().__init__()
        self.conv1 = nn.Conv2d(input_shape[0], 32, 8, 4)
        self.conv2 = nn.Conv2d(32, 64, 4, 2)
        self.conv3 = nn.Conv2d(64, 64, 3, 1)
        o = torch.zeros(1, *input_shape)
        o = F.relu(self.conv3(F.relu(self.conv2(F.relu(self.conv1(o))))))
        self.fc1 = nn.Linear(int(np.prod(o.shape[1:])), 512)
        self.fc2 = nn.Linear(512, n_actions)
    
    def forward(self, x):
        x = F.relu(self.conv1(x))
        x = F.relu(self.conv2(x))
        x = F.relu(self.conv3(x))
        x = F.relu(self.fc1(x.view(x.size(0), -1)))
        return self.fc2(x)

class DQNAgent:
    def __init__(self, config):
        self.config = config
        self.device = device
        self.env = gym.make(config.env_name, render_mode="rgb_array")
        self.n_actions = self.env.action_space.n
        self.preprocessor = FramePreprocessor()
        self.frame_stacker = FrameStacker()
        self.state_shape = (4, 84, 84)
        self.policy_net = DQN(self.state_shape, self.n_actions).to(device)
    
    def load_model(self, path):
        checkpoint = torch.load(path, map_location=self.device)
        self.policy_net.load_state_dict(checkpoint)
        print(f"✓ Loaded from {path}")



# Load
config = DQNConfig()
agent = DQNAgent(config)
agent.load_model("/kaggle/input/models/rohanprabhakar1103/atari-gammav1/other/default/1/gamma_0.95.pth")

# Play
def play(agent, episodes=5):
    agent.policy_net.eval()
    for ep in range(episodes):
        obs, _ = agent.env.reset()
        state = agent.frame_stacker.reset(agent.preprocessor.preprocess(obs))
        reward_total = 0
        
        while True:
            state_t = torch.FloatTensor(state).unsqueeze(0).to(agent.device)
            with torch.no_grad():
                action = agent.policy_net(state_t).argmax().item()
            obs, reward, term, trunc, _ = agent.env.step(action)
            state = agent.frame_stacker.add_frame(agent.preprocessor.preprocess(obs))
            reward_total += reward
            if term or trunc: break
        
        print(f"Episode {ep+1}: Reward = {reward_total}")

play(agent, episodes=5)

A.L.E: Arcade Learning Environment (version 0.11.2+ecc1138)
[Powered by Stella]


✓ Loaded from /kaggle/input/models/rohanprabhakar1103/atari-gammav1/other/default/1/gamma_0.95.pth
Episode 1: Reward = 1280.0
Episode 2: Reward = 600.0
Episode 3: Reward = 1780.0
Episode 4: Reward = 1040.0
Episode 5: Reward = 480.0


In [3]:

import matplotlib.pyplot as plt
import matplotlib.animation as animation
from IPython.display import HTML, display

def record_and_show(agent, max_steps=1000):
    """Record agent and display in notebook."""
    
    agent.policy_net.eval()
    
    obs, _ = agent.env.reset()
    state = agent.frame_stacker.reset(agent.preprocessor.preprocess(obs))
    
    frames = []
    total_reward = 0
    
    print("Recording gameplay...")
    
    for step in range(max_steps):
        # Save frame
        frames.append(agent.env.render())
        
        # Get action
        state_t = torch.FloatTensor(state).unsqueeze(0).to(agent.device)
        with torch.no_grad():
            action = agent.policy_net(state_t).argmax().item()
        
        # Step
        obs, reward, term, trunc, _ = agent.env.step(action)
        state = agent.frame_stacker.add_frame(agent.preprocessor.preprocess(obs))
        total_reward += reward
        
        if term or trunc:
            break
    
    print(f"✓ Recorded {len(frames)} frames, Reward: {total_reward}")
    
    fig, ax = plt.subplots(figsize=(6, 8))
    ax.axis('off')
    img = ax.imshow(frames[0])
    plt.close()
    
    def update(i):
        img.set_array(frames[i])
        return [img]
    
    ani = animation.FuncAnimation(fig, update, frames=len(frames), interval=50)
    
    # Also save as file
    ani.save('seaquest_gameplay.mp4', writer='ffmpeg', fps=30)
    print("✓ Saved as seaquest_gameplay.mp4")
    
    return total_reward

# Run it
record_and_show(agent, max_steps=1000)

Recording gameplay...
✓ Recorded 1000 frames, Reward: 720.0
✓ Saved as seaquest_gameplay.mp4


720.0

In [4]:
checkpoint = torch.load("/kaggle/input/models/rohanprabhakar1103/atari-gammav1/other/default/1/gamma_0.95.pth", map_location=device)
checkpoint.keys()

odict_keys(['conv1.weight', 'conv1.bias', 'conv2.weight', 'conv2.bias', 'conv3.weight', 'conv3.bias', 'fc1.weight', 'fc1.bias', 'fc2.weight', 'fc2.bias'])